In [5]:
import soundfile as sf
import sounddevice as sd
import whisper
import pyttsx3
import tempfile
from google import genai

In [1]:
import numpy as np

def sr_and_sw(sample_rate:int=16000, silence_threshold:float=0.01, silence_duration:float=2.0):
    print("Recording_started....")
    
    chunk_size = int(sample_rate * 0.1)  # 100ms chunks
    silence_chunks = int(silence_duration / 0.1)  # chunks needed for silence_duration
    
    recording = []
    silent_count = 0
    
    with sd.InputStream(samplerate=sample_rate, channels=1, blocksize=chunk_size, dtype=np.float32) as stream:
        while True:
            chunk, _ = stream.read(chunk_size)
            recording.append(chunk)
            
            # Check if chunk is silent
            if np.max(np.abs(chunk)) < silence_threshold:
                silent_count += 1
            else:
                silent_count = 0
            
            # Stop if silent for specified duration
            if silent_count >= silence_chunks:
                break
    
    print("Recording_stopped....")
    recording = np.concatenate(recording, axis=0).astype(np.float32)
    sf.write('recording.wav', recording, sample_rate)
    return recording

In [2]:
def transcribe_audio_to_text(audio):
    # with tempfile.NamedTemporaryFile(delete=False,suffix='mp3') as tempaudio:
    #     tempaudio.write(audio.read())
    #     temp_audio_path = tempaudio.name
        
    model = whisper.load_model('base',)
    text = model.transcribe(audio,language='en',task='transcribe')
    return text

In [6]:
engine = pyttsx3.init()
def generate_voice(text:str):
    engine.say(text)
    engine.runAndWait()

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key_llm = os.getenv("GEMINI_API_KEY")
def llm_output(text:str):
    client = genai.Client(api_key=api_key_llm)
    result = client.models.generate_content(model="gemini-2.0-flash",contents=text)
    return result.text

In [8]:
if __name__ == "__main__":
    print("program Started")
    sr_and_sw()
    audio_path = input("enter a audio path:")
    transcribe_audio = transcribe_audio_to_text(audio_path)
    print(transcribe_audio)
    llm_result = llm_output(transcribe_audio['text'])
    generate_voice(llm_result)

program Started
Recording_started....
Recording_stopped....


c:\Users\v.rajkumar\AppData\Local\Programs\Python\Python310\lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(

{'text': ' There is artificial intelligence and data science.', 'segments': [{'id': 0, 'seek': 0, 'start': 0.0, 'end': 3.0, 'text': ' There is artificial intelligence and data science.', 'tokens': [50364, 821, 307, 11677, 7599, 293, 1412, 3497, 13, 50514], 'temperature': 0.0, 'avg_logprob': -0.6355694424022328, 'compression_ratio': 0.8928571428571429, 'no_speech_prob': 0.05992720648646355}], 'language': 'en'}


KeyboardInterrupt: 